# Análisis de errores y sesgos

Clasificación cualitativa de los errores típicos de los mejores modelos.
Para 50 muestras del test set, cada predicción se analiza en busca de:

- **Alucinación:** información no presente en el artículo original.
- **Omisión:** hecho importante ausente del resumen.
- **Repetición:** contenido repetido dentro del resumen.
- **Error factual:** nombre, cifra o fecha incorrectos.
- **Sesgo de estilo:** lenguaje inapropiado, tono inadecuado, suposiciones.

Este análisis se automatiza parcialmente con Gemini (clasificación inicial)
y se revisa manualmente para los casos ambiguos.

In [ ]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import json
import re
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import google.generativeai as genai

from src.data.loader import load_config, load_cnn_dailymail
from src.models.loader import load_model, LoadedModel
from src.evaluation.inference import generate_summaries

# os.environ["GEMINI_API_KEY"] = "your-key-here"

cfg = load_config("../config/config.yaml")
dataset = load_cnn_dailymail(cfg)

TABLES_DIR = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Generate predictions from best T5 and Qwen3 checkpoints.
# ADJUST paths to match V3 winners.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

N_ANALYZE = 50
test_sub = dataset["test"].select(range(N_ANALYZE))
articles = list(test_sub["article"])
references = list(test_sub["highlights"])

# --- T5 ---
BEST_T5_DIR = "v3_t5_A"  # <-- CHANGE if needed
t5_ckpt = Path(f"../results/checkpoints/{BEST_T5_DIR}")
t5_subdirs = sorted([p for p in t5_ckpt.iterdir() if p.name.startswith("checkpoint-")])
t5_path = t5_subdirs[-1] if t5_subdirs else t5_ckpt

t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_path, dtype=torch.float32)
t5_tok = AutoTokenizer.from_pretrained(t5_path, use_fast=True)
t5_model.to("cuda").eval()
t5_loaded = LoadedModel(model=t5_model, tokenizer=t5_tok, name="google/flan-t5-base",
                         model_type="seq2seq", max_input_length=cfg["models"]["t5"]["max_input_length"])

t5_preds = generate_summaries(t5_loaded, articles, max_new_tokens=128, num_beams=4, batch_size=8)
del t5_model, t5_loaded; torch.cuda.empty_cache()

# --- Qwen3 ---
BEST_QWEN_DIR = "v3_qwen_A"  # <-- CHANGE if needed
qwen_base = load_model(cfg["models"]["qwen"])
qwen_ckpt = Path(f"../results/checkpoints/{BEST_QWEN_DIR}")
qwen_subdirs = sorted([p for p in qwen_ckpt.iterdir() if p.name.startswith("checkpoint-")])
qwen_path = qwen_subdirs[-1] if qwen_subdirs else qwen_ckpt
qwen_base.model = PeftModel.from_pretrained(qwen_base.model, str(qwen_path))
qwen_base.model.eval(); qwen_base.model.config.use_cache = True

qwen_preds = generate_summaries(qwen_base, articles, max_new_tokens=128, num_beams=4, batch_size=2)
del qwen_base; torch.cuda.empty_cache()

print(f"Generated {len(t5_preds)} T5 and {len(qwen_preds)} Qwen3 predictions.")

In [ ]:
# Automated error classification with Gemini
ERROR_PROMPT = """\
You are an expert evaluator of news summaries. Given the original article \
and a machine-generated summary, classify ALL errors present in the summary.

Error categories:
- HALLUCINATION: summary contains information not in the article
- OMISSION: summary misses a key fact from the article
- REPETITION: same content appears multiple times in the summary
- FACTUAL_ERROR: names, numbers, dates are wrong
- STYLE_BIAS: inappropriate tone, assumptions, or biased language
- NONE: no errors detected

Respond ONLY with valid JSON (no markdown fences):
{{"errors": [{{"type": "<category>", "detail": "<brief description>"}}]}}

If there are no errors, respond: {{"errors": [{{"type": "NONE", "detail": "No errors detected"}}]}}

---
ARTICLE:
{article}

---
SUMMARY:
{summary}
"""

def classify_errors(gemini_model, article, summary, max_retries=2):
    """Classify errors in a single summary using Gemini."""
    prompt = ERROR_PROMPT.format(article=article[:3000], summary=summary)
    for attempt in range(max_retries + 1):
        try:
            resp = gemini_model.generate_content(prompt)
            text = resp.text.strip()
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
            return json.loads(text)["errors"]
        except Exception as e:
            if attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            return [{"type": "PARSE_ERROR", "detail": str(e)}]

# Initialize Gemini
api_key = os.environ.get("GEMINI_API_KEY", "")
genai.configure(api_key=api_key)
gemini = genai.GenerativeModel("gemini-2.0-flash")

# Classify errors for both models
all_errors = []
for model_name, preds in [("Flan-T5-base", t5_preds), ("Qwen3-1.7B", qwen_preds)]:
    print(f"\nClassifying errors for {model_name}...")
    for i, (art, pred) in enumerate(zip(articles, preds)):
        errors = classify_errors(gemini, art, pred)
        for err in errors:
            all_errors.append({"model": model_name, "index": i,
                             "error_type": err["type"], "detail": err["detail"]})
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(preds)}")
        time.sleep(0.5)

df_errors = pd.DataFrame(all_errors)
df_errors.to_csv(TABLES_DIR / "error_analysis.csv", index=False)
print(f"\nTotal error annotations: {len(df_errors)}")

In [ ]:
# Error distribution by model and type
# Filter out NONE and PARSE_ERROR
real_errors = df_errors[~df_errors["error_type"].isin(["NONE", "PARSE_ERROR"])]

fig, ax = plt.subplots(figsize=(9, 5))
ct = pd.crosstab(real_errors["error_type"], real_errors["model"])
ct.plot(kind="barh", ax=ax)
ax.set_title("Error distribution by model and type")
ax.set_xlabel("Count (across 50 samples)")
ax.set_ylabel("Error type")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "error_distribution.png", bbox_inches="tight", dpi=120)
plt.show()

# Summary table
print("\nError counts per model:")
print(ct.to_string())

In [ ]:
# Percentage of samples with at least one error (excluding NONE)
for model in ["Flan-T5-base", "Qwen3-1.7B"]:
    model_errors = real_errors[real_errors["model"] == model]
    samples_with_errors = model_errors["index"].nunique()
    pct = samples_with_errors / N_ANALYZE * 100
    print(f"{model}: {samples_with_errors}/{N_ANALYZE} samples with errors ({pct:.0f}%)")

# Show examples of hallucinations
hallucinations = real_errors[real_errors["error_type"] == "HALLUCINATION"]
print(f"\nHallucinations found: {len(hallucinations)}")
for _, row in hallucinations.head(3).iterrows():
    idx = row["index"]
    model = row["model"]
    pred = t5_preds[idx] if "T5" in model else qwen_preds[idx]
    print(f"\n[{model}] idx={idx}: {row['detail']}")
    print(f"Summary: {pred[:200]}")
    print("-" * 60)

## Análisis de errores y sesgos

*(Rellenar tras ejecución. Puntos a cubrir:)*

1. **Tipo de error más frecuente por modelo** — ¿T5 alucina más que Qwen3?
   ¿Qwen3 omite más información (por parafraseo excesivo)?
2. **Tasa de error global** — ¿qué porcentaje de resúmenes tiene al menos
   un error detectable?
3. **Sesgos detectados** — ¿hay patrones en el tipo de artículos o temas
   donde los modelos fallan más?
4. **Limitaciones del análisis automático** — Gemini puede no detectar
   alucinaciones sutiles o errores factuales que requieran conocimiento
   del dominio.